In [262]:
from pathlib import Path

ROOT = Path("./results")    # 根目录：包含 gcd/ 和 openset/
SETTING = "gcd"             # 在 "gcd" 与 "openset" 间切换

# 每个 setting 对应要读取的任务名；留空 [] 或 ["*"] 表示扫描该 setting 下所有含 results.csv 的任务
TASKS_MAP = {
    "gcd":     ["alup", "deepaligned", "dpn", "geoid", "loop", "sdc", "tan"],
    "openset": ["ab", "adb", "deepunk", "doc", "dyen", "knncon", "plm_ood", "scl"],
}
METHOD_RENAME = {
    # gcd
    "deepaligned": "DeepAligned",
    "geoid": "GeoID",
    "sdc": "SDC",
    "dpn": "DPN",
    "tan": "TAN",
    "loop": "LOOP",
    "alup": "ALUP",

    # openset
    "doc": "DOC",
    "deepunk": "DeepUNK",
    "adb": "ADB",
    "scl": "SCL",
    "ab": "AB",
    "knncon": "KNNCon",
    "knn_con": "KNNCon",
    "dyen": "DyEn",
    "plm_ood": "LLM_OOD",
}

# 每个 setting 对应要展示的指标（按截图风格）
METRICS_MAP = {
    # "gcd":     ["K-ACC", "N-ACC"],
    "gcd":     ["N-ACC"],
    # "openset": ["K-F1", "N-F1"],
    "openset": ["N-F1"],
}

LCR_KEEP = {"gcd": 0.1, "openset": 0.5} # 过滤的 labeled_ratio 
KCR_ORDER = [0.25, 0.50, 0.75]

ROUND_DECIMALS = 2          # 数值保留小数位
OUTPUT_TEX = f"{METRICS_MAP['gcd'][0]}_{SETTING}_{LCR_KEEP['gcd']}_wide_table.tex"  # 输出的 LaTeX 文件名
HIGHLIGHT_TOP2 = True       # 是否加粗最佳、下划线次佳

# 指定每个 setting 的 Method 排序；未列出的模型会排在这些之后（按字母序）
METHOD_ORDER_MAP = {
    "gcd": ["DeepAligned", "GeoID", "SDC", "DPN", "TAN", "LOOP", "ALUP"],
    "openset": ["DOC", "DeepUNK", "ADB", "SCL", "AB", "KNNCon", "DyEn", 
                "MaxSoftmax", "OpenMax", "TemperatureScaling", "Mahalanobis", 
                "EnergyBased", "LogitNorm", "Entropy", "KLMatching", "MaxLogit"
            ], 
}
# METHOD_ORDER_MAP = {
#     "gcd": ["DeepAligned", "GeoID", "SDC", "DPN", "TAN", "LOOP", "ALUP"],
#     "openset": ["DOC", "DeepUNK", "ADB", "SCL", "AB", "KNNCon", "DyEn", 
#                  "OpenMax", "EnergyBased",  "MaxLogit"
#             ], 
# }
# METHOD_ORDER_MAP = {
#     "gcd": ["DeepAligned", "GeoID", "SDC", "DPN", "TAN", "LOOP", "ALUP"],
#     "openset": ["DyEn"], 
# }



In [263]:
# ====== 读取到生成 LaTeX（行索引：Metric -> KCR -> Method；列：Datasets） ======
import pandas as pd
import numpy as np
import json
import ast

def parse_args_field(s: str) -> dict:
    """尽量鲁棒地把CSV里的args字符串解析成dict。"""
    if pd.isna(s):
        return {}
    txt = str(s)
    # 先尝试json
    try:
        return json.loads(txt)
    except Exception:
        pass
    # CSV里常见的双引号转义：{""k"":""v""}
    try:
        fixed = txt.replace('""', '"')
        # 去掉可能包裹的一对首尾引号
        if len(fixed) >= 2 and fixed[0] == '"' and fixed[-1] == '"':
            fixed = fixed[1:-1]
        return json.loads(fixed)
    except Exception:
        pass
    # 再兜底用 ast（容忍单引号等）
    try:
        return ast.literal_eval(txt)
    except Exception:
        return {}


base = ROOT / SETTING
metrics = METRICS_MAP[SETTING]            # {'gcd': ['K-ACC','N-ACC'], 'openset': ['K-F1','N-F1']}
KEEP_LCR_VALUE = LCR_KEEP[SETTING]        # {'gcd': 0.1, 'openset': 1.0}

# 只展示这些 dataset（按顺序）；不存在的会自动跳过
# DATASETS_WHITELIST = [
#     "20NG", "DBPedia", "TREC", "Yahoo",
#     "banking", "clinc", "ecdt", "hwu", "mcid",
#     "news", "stackoverflow", "thucnews",
# ]
DATASETS_WHITELIST = [
    "20NG", "thucnews", "Yahoo", "TREC", 
    "banking", "stackoverflow", "clinc", "hwu", "ecdt", "mcid",
    "DBPedia", 
]

def _normalize_key(s: str) -> str:
    return str(s).strip().lower().replace("_", "").replace("-", "")

def rename_method(raw_name: str, fallback_task: str) -> str:
    key = _normalize_key(raw_name if pd.notnull(raw_name) else fallback_task)
    return METHOD_RENAME.get(key, str(raw_name if pd.notnull(raw_name) else fallback_task))


# 1) 任务列表
task_list = TASKS_MAP.get(SETTING, [])
if (not task_list) or (task_list == ["*"]):
    task_list = sorted(p.name for p in base.iterdir()
                       if p.is_dir() and (p / "results.csv").exists())
if not task_list:
    raise FileNotFoundError(f"未发现任务目录：{base}")

# # 调试############################
# print(f"[SCAN] SETTING={SETTING}, base={base}")
# for t in task_list:
#     print(f"[SCAN] 发现任务目录：{t} -> {(base/t/'results.csv').resolve()}")
# ############################

# 2) 读取并合并（对齐第三个表：读失败就跳过该任务）
frames, missing = [], []
ALIASES_PLM = {"plmood", "llmood"}  # 与第三个表一致的别名集合

for task in task_list:
    csv_path = base / task / "results.csv"
    try:
        tdf = pd.read_csv(csv_path)
        # print(f"[OK] 读取成功：{csv_path}  shape={tdf.shape}")
    except Exception as e:
        # 与第三个表一致：这类坏CSV直接跳过，不让整个流程崩
        print(f"[WARN] 读取失败，已跳过：{csv_path}  |  错误：{type(e).__name__}: {e}")
        missing.append(str(csv_path))
        continue

    # 解析 args（后面要用 fold_type / Detector）
    tdf["__args_obj__"] = tdf["args"].apply(parse_args_field)

    # 统一 Method：普通方法重命名；plm_ood → 用 Detector/Detecor
    if "method" in tdf.columns:
        def _method_from_row(row):
            raw_m = row.get("method", None)
            renamed = rename_method(raw_m, task)
            # 是否属于 plm_ood（含别名）
            if any(_normalize_key(x) in ALIASES_PLM for x in (task, raw_m, renamed)):
                det = row["__args_obj__"].get("Detector") or row["__args_obj__"].get("Detecor")
                return det if det else "LLM-OOD"
            return renamed
        tdf["Method"] = tdf.apply(_method_from_row, axis=1)
    else:
        if _normalize_key(task) in ALIASES_PLM:
            tdf["Method"] = tdf["__args_obj__"].apply(
                lambda d: d.get("Detector") or d.get("Detecor") or "LLM-OOD"
            )
        else:
            tdf["Method"] = rename_method(task, task)

    # 基本字段
    tdf["dataset"] = tdf["dataset"].astype(str)
    tdf["KCR"] = tdf.get("known_cls_ratio", np.nan)
    tdf["LCR"] = tdf.get("labeled_ratio", np.nan)

    # —— 打印这个文件里最终 Method 的分布（重命名之后）
    vc = tdf["Method"].astype(str).value_counts(dropna=False).to_dict()
    print(f"[INFO] {csv_path.name} -> Method分布：{vc}")

    # 清理临时列
    tdf.drop(columns=["__args_obj__"], inplace=True, errors="ignore")

    frames.append(tdf)

if not frames:
    raise FileNotFoundError("未找到任何 results.csv；失败文件：\n" + "\n".join(missing))
df = pd.concat(frames, ignore_index=True)

print(f"[INFO] 合并后总行数：{len(df)}")
print(f"[INFO] 合并后 Method 去重：{sorted(df['Method'].astype(str).unique().tolist())}")

# ALLOWED = set(METHOD_ORDER_MAP.get(SETTING, []))
# if ALLOWED:
#     df = df[df["Method"].isin(ALLOWED)].copy()
# —— 仅保留 METHOD_ORDER_MAP 里列出的模型（白名单）
ALLOWED = [m for m in METHOD_ORDER_MAP.get(SETTING, []) if m]  # 去掉空值
if ALLOWED:
    df["Method"] = df["Method"].astype(str)
    df = df[df["Method"].isin(set(ALLOWED))].copy()
    if df.empty:
        raise ValueError(
            f"{SETTING}: 方法白名单过滤后无数据。请检查 METHOD_ORDER_MAP['{SETTING}'] "
            f"是否与实际 Method 名（含各 Detector 名）一致。"
        )
    # >>> 调试打印：白名单过滤后的方法列表与计数
    remaining = sorted(df["Method"].unique().tolist())
    print(f"[DEBUG] 白名单过滤后 Methods: {remaining}")
    print("[DEBUG] 白名单过滤后各方法行数：")
    print(df["Method"].value_counts().sort_index())

    # 哪些在白名单里但没出现
    missing_allowed = [m for m in ALLOWED if m not in remaining]
    if missing_allowed:
        print(f"[WARN] 白名单中未出现的方法：{missing_allowed}")

# 3) 只留需要字段 + 过滤 LCR
required_cols = ["dataset", "Method", "KCR", "LCR"] + metrics
miss = [c for c in required_cols if c not in df.columns]
if miss:
    raise KeyError(f"以下列缺失：{miss}")

df = df[df["LCR"] == KEEP_LCR_VALUE].copy()
if df.empty:
    raise ValueError(f"{SETTING}: 过滤 LCR={KEEP_LCR_VALUE} 后无数据。")

# 4) 聚合到 (dataset, Method, KCR, LCR)
grouped = (
    df[required_cols]
    .groupby(["dataset", "Method", "KCR", "LCR"], as_index=False)
    .mean(numeric_only=True)
)

# 5) 长表：把 metric 展为行维度
long = grouped.melt(
    id_vars=["dataset", "Method", "KCR", "LCR"],
    value_vars=metrics, var_name="Metric", value_name="value"
)

# >>> 在这里插入：openset 的指标若是分数(<1)，自动转为百分数 <<<
if SETTING == "openset":
    mask = long["Metric"].isin(["K-F1", "N-F1", "ACC", "F1"])  # 自行选择
    long.loc[mask, "value"] = long.loc[mask, "value"].apply(
        lambda v: (v * 100.0) if (pd.notnull(v) and isinstance(v, (int, float)) and v < 1.0) else v
    )


# 6) 只保留白名单 dataset（大小写不敏感）
wl = [d.lower() for d in DATASETS_WHITELIST]
long = long[long["dataset"].str.lower().isin(wl)]
if long.empty:
    raise ValueError("白名单筛选后无可用 dataset，请检查名称大小写。")

# 7) 透视：行索引 (Metric, KCR, Method)，列是 dataset（单层）
pivot = long.pivot_table(
    index=["Metric", "KCR", "Method"],
    columns="dataset",
    values="value",
    aggfunc="mean"
)

# 8) 行排序：Metric 按 metrics 顺序；KCR 按 KCR_ORDER；Method 按自定义顺序（未指定的排后，字母序）
pivot = pivot.sort_index(level=[0,1,2])
idx = pivot.index

metric_cat = pd.Categorical(
    idx.get_level_values(0),
    categories=metrics,
    ordered=True
)

kcr_cat = pd.Categorical(
    idx.get_level_values(1),
    categories=KCR_ORDER or sorted(idx.get_level_values(1).unique()),
    ordered=True
)

# 计算 Method 排序
methods_present = list(pivot.index.get_level_values(2).unique())
whitelist = METHOD_ORDER_MAP.get(SETTING, []) or []
# 只按白名单顺序展示，未列出的不显示（前面已过滤，这里再保险按交集排）
method_order = [m for m in whitelist if m in methods_present]

method_cat = pd.Categorical(
    idx.get_level_values(2),
    categories=method_order,
    ordered=True
)

new_index = pd.MultiIndex.from_arrays(
    [metric_cat, kcr_cat, method_cat],
    names=idx.names
)
pivot = pivot.reindex(new_index).sort_index(level=[0,1,2])

# 9) 列排序：严格按白名单顺序，且只保留真实存在的列
existing_cols = [c for c in pivot.columns.tolist() if str(c).lower() in wl]
datasets_order = [d for d in DATASETS_WHITELIST if d.lower() in [str(c).lower() for c in existing_cols]]
pivot = pivot.reindex(columns=datasets_order)

# 10) 数值
pivot = pivot.round(ROUND_DECIMALS)

# 11) 高亮：现在每个 dataset 是一个独立的列，按 (Metric, KCR) 对“Method”排名
def highlight_col(col_ser: pd.Series) -> pd.Series:
    # col_ser 是一个以 (Metric, KCR, Method) 为索引的单列
    out = col_ser.astype(object)
    df_float = pd.to_numeric(col_ser, errors="coerce")
    # 按 (Metric, KCR) 分组
    for (metric_val, kcr_val), sub in df_float.groupby(level=[0,1]):
        order = sub.sort_values(ascending=False, kind="mergesort")
        if len(order) >= 1 and pd.notnull(order.iloc[0]):
            out.loc[(metric_val, kcr_val, order.index.get_level_values(2)[0])] = (
                "\\textbf{" + f"{order.iloc[0]:.{ROUND_DECIMALS}f}" + "}"
            )
        if len(order) >= 2 and pd.notnull(order.iloc[1]):
            out.loc[(metric_val, kcr_val, order.index.get_level_values(2)[1])] = (
                "\\underline{" + f"{order.iloc[1]:.{ROUND_DECIMALS}f}" + "}"
            )
        # 其它常规格式
        for idxi, v in sub.items():
            if pd.isnull(v):
                out.loc[idxi] = ""
            elif out.loc[idxi] == col_ser.loc[idxi]:
                out.loc[idxi] = f"{v:.{ROUND_DECIMALS}f}"
    return out

formatted = pd.DataFrame(index=pivot.index)
for ds in datasets_order:
    formatted[ds] = highlight_col(pivot[ds]) if ds in pivot.columns else ""

# 12) 组装 LaTeX
# ====== 12) 组装成你给的标准 LaTeX 版式（table* + tabularx + 多行合并 + Avg 列）======

from math import isnan

# --- 数据集显示名（与示例一致） ---
DISPLAY_NAME_MAP = {
    "20ng": "20NG",
    "thucnews": "THUCNews",
    "yahoo": "Yahoo",
    "trec": "TREC",
    "banking": "BANK",
    "stackoverflow": "S.O.",
    "clinc": "CLINC",
    "hwu": "HWU",
    "ecdt": "ECDT",
    "mcid": "M-CID",
    "dbpedia": "DBPedia",
}

def display_ds_name(raw: str) -> str:
    key = str(raw).strip().lower()
    return DISPLAY_NAME_MAP.get(key, raw)

# 计算每个 (Metric, KCR) 下的方法行数（用于 \multirow）
group_sizes = formatted.groupby(level=[0,1]).size().to_dict()

# # 反查：某个 (Metric,KCR) 起始行在 formatted 的第几行（用于放置 \multirow 起始）
# # 我们用遍历时的“第一行”作为起始
# first_row_flags = {}
# seen_pairs = set()
# for idx in formatted.index:
#     pair = (idx[0], idx[1])
#     if pair not in seen_pairs:
#         first_row_flags[pair] = True
#         seen_pairs.add(pair)
#     else:
#         first_row_flags[pair] = False

# 用于 \cmidrule 的列范围：总列数 = 3(左) + len(datasets) + 1(Avg)
total_cols = 3 + len(datasets_order) + 1
cmidrule_cols = f"2-{total_cols}"

# LaTeX 文字转义（沿用你之前的函数）
def latex_escape_text(x) -> str:
    s = str(x)
    rep = {
        "\\": r"\textbackslash{}",
        "&":  r"\&",
        "%":  r"\%",
        "$":  r"\$",
        "#":  r"\#",
        "_":  r"\_",
        "{":  r"\{",
        "}":  r"\}",
        "~":  r"\textasciitilde{}",
        "^":  r"\textasciicircum{}",
    }
    return "".join(rep.get(ch, ch) for ch in s)

# 计算某一行（Metric,KCR,Method）在“数值型 pivot”上的均值（Avg），忽略缺失
def row_avg_numeric(metric, kcr, method) -> float | None:
    try:
        row = pivot.loc[(metric, kcr, method)]
    except KeyError:
        return None
    vals = []
    for ds in datasets_order:
        if ds in row.index:
            v = row[ds]
            if pd.notnull(v):
                vals.append(float(v))
    return (sum(vals) / len(vals)) if vals else None

# 1) 列头：Metric & KCR & Method & 11 个数据集（显示名） & 粗体 Avg.
header_cells = ["Metric", "KCR", "Method"]
for ds in datasets_order:
    header_cells.append(latex_escape_text(display_ds_name(ds)))
header_cells.append(r"\textbf{Avg.}")
header_line = " & ".join(header_cells) + r" \\ \midrule"


def _fmt_kcr(x):
    if pd.isnull(x): 
        return ""
    return f"{float(x):.2f}".rstrip("0").rstrip(".")
    


lines = []
last_pair = (None, None)

for (metric, kcr, method), row in formatted.iterrows():
    pair = (metric, kcr)
    is_new_pair = (pair != last_pair)           # ← 这行替代 first_row_flags

    # 不同 (Metric,KCR) 段之间画线
    if last_pair != (None, None) and is_new_pair:
        lines.append(fr"\cmidrule(l){{{cmidrule_cols}}}")

    cells = []

    # Metric 列 multirow：每个 Metric 第一次出现时写
    if last_pair[0] != metric:
        metric_total_rows = sum(sz for (m, _k), sz in group_sizes.items() if m == metric)
        cells.append(fr"\multirow{{{metric_total_rows}}}{{*}}{{{metric}}}")
    else:
        cells.append("")

    # KCR 列 multirow：每个 (Metric,KCR) 段的第一行写
    if is_new_pair:
        kcr_rows = group_sizes[pair]
        cells.append(fr"\multirow{{{kcr_rows}}}{{*}}{{{_fmt_kcr(kcr)}}}")
    else:
        cells.append("")

    # Method 列
    cells.append(latex_escape_text(method))

    # 数据集列（保留你的高亮）
    for ds in datasets_order:
        val = row.get(ds, "")
        if isinstance(val, str):
            cells.append(val)
        else:
            cells.append("" if pd.isnull(val) else f"{val:.{ROUND_DECIMALS}f}")

    # Avg 列
    avg_v = row_avg_numeric(metric, kcr, method)
    cells.append("" if (avg_v is None or (isinstance(avg_v, float) and np.isnan(avg_v)))
                 else f"{avg_v:.{ROUND_DECIMALS}f}")

    lines.append(" & ".join(cells) + r" \\")
    last_pair = pair    

# 3) 组合为完整 LaTeX（与你提供的模板完全一致；caption 先用 XXX）
# 列规格：@{}l l l |l *{10}{Y}|Y@{}  —— 3 个左对齐 + 1 个 l + 10 个 Y + 1 个 Y(Avg)
colspec = "@{}l l l |l *{10}{Y}|Y@{}"

latex_body = "\n".join(lines)

latex_table = "\n".join([
    r"\begin{table*}[ht!]",
    r"\caption{OSTC, ACC, LAR=0.1}",
    rf"\label{{table::{SETTING}_main}}",
    r"\centering",
    r"\tiny",
    r"\setlength{\tabcolsep}{4pt}",
    "",
    r"% Y 列型：可在此处定义（若已在导言区定义，可删掉下一行）",
    r"\newcolumntype{Y}{>{\centering\arraybackslash}X}",
    r"\begin{tabularx}{\textwidth}{" + colspec + r"}",
    r"\toprule",
    header_line,
    latex_body,
    r"\bottomrule",
    r"\end{tabularx}",
    r"\end{table*}",
])

Path(OUTPUT_TEX).write_text(latex_table, encoding="utf-8")
print(f"[OK] 生成（标准版式）：{Path(OUTPUT_TEX).resolve()}")


[INFO] results.csv -> Method分布：{'ALUP': 908}
[INFO] results.csv -> Method分布：{'DeepAligned': 903}
[INFO] results.csv -> Method分布：{'DPN': 843}
[INFO] results.csv -> Method分布：{'GeoID': 872}
[INFO] results.csv -> Method分布：{'LOOP': 846}
[INFO] results.csv -> Method分布：{'SDC': 860}
[INFO] results.csv -> Method分布：{'TAN': 2459}
[INFO] 合并后总行数：7691
[INFO] 合并后 Method 去重：['ALUP', 'DPN', 'DeepAligned', 'GeoID', 'LOOP', 'SDC', 'TAN']
[DEBUG] 白名单过滤后 Methods: ['ALUP', 'DPN', 'DeepAligned', 'GeoID', 'LOOP', 'SDC', 'TAN']
[DEBUG] 白名单过滤后各方法行数：
Method
ALUP            908
DPN             843
DeepAligned     903
GeoID           872
LOOP            846
SDC             860
TAN            2459
Name: count, dtype: int64
[OK] 生成（标准版式）：/ssd/code/bolt/N-ACC_gcd_0.1_wide_table.tex


/tmp/ipykernel_3811099/4087731443.py:246: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for (metric_val, kcr_val), sub in df_float.groupby(level=[0,1]):
/tmp/ipykernel_3811099/4087731443.py:246: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for (metric_val, kcr_val), sub in df_float.groupby(level=[0,1]):
/tmp/ipykernel_3811099/4087731443.py:246: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for (metric_val, kcr_val), sub